# ComplaintIQ - trivial & heuristic baselines (`04a_baselines_trivial`)

The first modeling notebook. Before any learned model, we establish the **floor**: the
scores a trivial or rules-only approach reaches, so later notebooks have an honest bar to
beat. **No feature engineering** - raw columns only.

Two baselines here, both for the supervised target `monetary_relief`:
- **Majority class** ("predict never"): the accuracy trap, and why ranking metrics matter.
- **Product-bucket heuristic**: rank complaints by their `product`'s historical relief
  rate. This is the baseline the proposal's lift claim is measured against (computed in
  `02_eda_supervised.ipynb` section 9a); `04b` and later models must beat it.

Simple logistic-regression baselines live in **`04b_baselines_logreg.ipynb`**;
unsupervised baselines in **`05_baselines_unsupervised.ipynb`**.

## How to read this notebook
Baselines are evaluated the way the product will be: a **chronological split** (train on
older complaints, test on the most recent months) so scores reflect deployment and do not
leak, and **imbalance-aware metrics** (PR-AUC, ROC-AUC, lift at top-1% / 5% / 10%, Brier), never raw
accuracy. All heavy data work runs in Spark; only a stratified sample is pulled into
pandas for the sklearn baselines.

> **Note:** the target is rare (~1.28% base rate, July 2026 snapshot), so a stratified
> sample keeps the positive class well represented while the reported metrics preserve the
> true base rate.

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from pyspark.sql import DataFrame as SparkDataFrame
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.metrics import report, top_k_lift
from complaintiq.sampling import stratified_pandas

# The tools we lean on across the project.
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# House plotting theme: clean grid + colorblind-safe palette, used everywhere.
sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

# The one true seed, so any sample below is reproducible run-to-run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("seaborn:", sns.__version__)

---
## 1. Load + chronological split *(shared modeling foundation)*

Load intake metadata and the target in Spark, then split by time at the 80th percentile of
`date_received`: older complaints train, the most recent months test. `complaint_text` is
left on disk here (metadata baselines only); `04b` loads narratives for the TF-IDF baseline.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

# Resolve the data directory: the UC volume when running on Databricks,
# else the local ../data produced by `make parquet`.
VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"
if not path.exists():
    raise FileNotFoundError("data/complaints.parquet not found. Run `make parquet` first.")

# Raw intake columns only - no engineered features.
META_COLS = ["product", "sub_product", "issue", "sub_issue", "submitted_via", "state", "tags"]
spark_df = (
    spark.read.parquet(str(path))
    .select("monetary_relief", F.to_date("date_received").alias("date_received"), *META_COLS)
    .dropna(subset=["date_received"])
)

# Chronological split at the 80th percentile of the date (ranked on epoch-days, since
# approxQuantile needs a numeric column).
spark_df = spark_df.withColumn("epoch", F.datediff("date_received", F.lit("1970-01-01")))
cut = spark_df.approxQuantile("epoch", [0.80], 0.001)[0]
train_sdf = spark_df.filter(F.col("epoch") <= cut).drop("epoch")
test_sdf = spark_df.filter(F.col("epoch") > cut).drop("epoch")
print(f"train rows: {train_sdf.count():,}  |  test rows: {test_sdf.count():,}")

---
## 2. Stratified sample into pandas

sklearn works in-memory, so draw a fixed-seed sample from each split. Stratify on
`monetary_relief` (**proportional**, same fraction per class) so the rare positive class is
represented without distorting the base rate the metrics depend on.

In [ ]:
train = stratified_pandas(train_sdf, 300_000)
test = stratified_pandas(test_sdf, 300_000)
y_train = train["monetary_relief"].to_numpy()
y_test = test["monetary_relief"].to_numpy()
base_rate = y_test.mean()
print(f"train sample: {len(train):,} (pos {y_train.mean():.4%})")
print(f"test sample:  {len(test):,} (pos {base_rate:.4%})")

---
## 3. Evaluation helpers

One place defines how every baseline (here and in `04b`) is scored, so numbers are
comparable across notebooks: PR-AUC and ROC-AUC over the whole test set, plus the
operational **precision and lift at the top 1% / 5% / 10%** an analyst queue actually sees. top-10% lift is capped at 10x by construction (lift = precision / base rate, precision <= 1, so <= 1/k), so a smaller queue has a higher ceiling and shows ranking power the 10% slice cannot express.

In [ ]:
results = []

---
## 4. Baseline A - majority class ("predict never")

The trivial baseline predicts `monetary_relief = 0` for everyone (probability 0). It shows
why accuracy is the wrong metric: it scores near-perfect accuracy while being useless for a
ranked queue.

In [ ]:
scores_majority = np.zeros(len(test))  # everyone gets probability 0
accuracy = (y_test == 0).mean()
print(f"'predict never' accuracy: {accuracy:.4%}  <- looks great, ranks nothing")
results.append(report("majority_class", y_test, scores_majority))

> **What you're seeing:** a model that never predicts relief.
>
> **Notice:** accuracy is ~98.7%, but the ranking metrics show no skill: PR-AUC equals the
> base rate (a precision-recall curve with no ranking ability sits at the positive fraction),
> ROC-AUC is 0.5 by convention (undefined for a constant score), and lift is ~1x at every queue size (top-1% / 5% / 10%) - a constant score has no ranking power at any depth.
>
> **Why it matters:** this is the accuracy trap the proposal calls out. Every useful model
> must be judged on PR-AUC / lift, not accuracy.

---
## 5. Baseline B - product-bucket heuristic

The rules-only baseline: learn each `product`'s relief rate on the **training** window,
score test complaints by their product's rate (unseen products get the train base rate),
and rank. No model, no text - just the single most predictive raw column. This is the
number `02` section 9a computed and the bar the learned models in `04b` must beat.

In [ ]:
# Learn product relief rates on TRAIN only (no leakage), score TEST by lookup.
prod_rate = train.groupby("product", observed=True)["monetary_relief"].mean()
train_base = y_train.mean()
scores_bucket = test["product"].map(prod_rate).fillna(train_base).to_numpy()
results.append(report("product_bucket", y_test, scores_bucket))

> **What you're seeing:** ranking by a product's historical relief rate.
>
> **Notice:** a strong lift from one raw column and zero learning - ~36x at top-1%, ~19x at top-5%, ~9.7x at top-10% (the top-10% number is near its 10x ceiling).
>
> **Why it matters:** this is the real bar. "Useful" means beating this, not the base rate - 
> a fixed "3-5x" target would be *worse* than this heuristic.

---
## 6. Baselines summary

The scoreboard `04b` and later notebooks extend. The product-bucket lift is the line to beat.

In [ ]:
board = pd.DataFrame(results).set_index("model")
display(board.round(4))
ax = sns.barplot(
    x=board["lift"], y=board.index, hue=board.index, palette="colorblind", legend=False
)
ax.set_xlabel("top-10% lift over base rate")
ax.set_ylabel("")
ax.set_title("Baseline lift (the bar to beat)")
for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.2f}x",
        (p.get_width(), p.get_y() + p.get_height() / 2),
        ha="left",
        va="center",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

---
## 7. Takeaways

> **What these baselines establish:**
> - **Accuracy is out:** the majority-class model proves it - near-perfect accuracy, useless
>   ranking. Report PR-AUC / ROC-AUC / lift instead.
> - **The bar is the product-bucket heuristic** - ~36x / 19x / 9.7x lift at top-1% / 5% / 10%, not the raw base rate; beat it at the queue size a reviewer actually works.
> - **Method for every model:** chronological split, proportional stratified sample, the
>   shared `report()` metrics - so `04b` and `05` are directly comparable.

Continue in **`04b_baselines_logreg.ipynb`** (simple logistic regression on raw metadata and
raw TF-IDF text) and **`05_baselines_unsupervised.ipynb`** (baseline clustering).